In [ ]:
!pip install transformers accelerate peft datasets bitsandbytes torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"  # или твоя модель

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

In [ ]:
# Загрузи синтетический датасет из src/evaluation/sample_qa.json
import json
with open("../src/evaluation/sample_qa.json") as f:
    data = json.load(f)

# Формат: вопрос-ответ с контекстом
formatted = []
for item in data:
    text = f"[INST] Используя контекст, ответь на вопрос: {item['question']} [/INST] {item['answer']}"
    formatted.append(text)

# Токенизация
def tokenize_function(examples):
    return tokenizer(examples, padding="max_length", truncation=True, max_length=512)

from datasets import Dataset
dataset = Dataset.from_dict({"text": formatted})
tokenized = dataset.map(tokenize_function, batched=True)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./qlora_finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-4,
    bf16=True,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
)
trainer.train()
model.save_pretrained("./qlora_adapter")